# Lab 3.4 &mdash; Checkpointing &mdash; Resume, Approve, Rewind, Audit

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; Memory, State &amp; the LangGraph Substrate**

### What you'll do
- Write a checkpointer that persists state after every node
- Kill a run mid-flight and resume it without repeating work
- Pause before an irreversible step and wait for approval
- Rewind to an earlier checkpoint, change one field, and re-run
- Read the checkpoint history as an audit trail

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Builds directly on Lab 3.3's graph.** One mechanism -- state written after every
> node -- gives you all four capabilities.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# These are the tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

After every node, write the whole state under a **thread id**. That single mechanism gives you
resume after a crash, human-in-the-loop approval, time travel, and an audit trail &mdash; the last one
as a by-product rather than a feature anyone built.

It matters that the trail is **recorded**, not generated: it is what the system actually held, not
the model's account of what it did.

## Section 1 &mdash; A checkpointer

Append-only, keyed by thread. Simple enough to fit on a slide; the real one differs mainly in where
it writes.

In [ ]:
class Checkpointer:
    """Append-only state history per thread. A real one writes to Postgres or Redis."""

    def __init__(self):
        self.threads: dict[str, list[dict]] = {}

    def put(self, thread: str, node: str, state: dict) -> None:
        """Record the state as it stands AFTER `node` ran."""
        self.threads.setdefault(thread, []).append(
            {"seq": len(self.threads.get(thread, [])), "after": node,
             "state": json.loads(json.dumps(state))})     # a snapshot, not a reference

    def latest(self, thread: str) -> dict | None:
        """The most recent checkpoint, or None if the thread is new."""
        history = self.threads.get(thread, [])
        return BLANK                 # TODO: the last checkpoint, or None when there is none

    def at(self, thread: str, seq: int) -> dict | None:
        for cp in self.threads.get(thread, []):
            if cp["seq"] == seq:
                return cp
        return None

    def history(self, thread: str) -> list[dict]:
        return list(self.threads.get(thread, []))

In [ ]:
# --- Self-check: Section 1
def _cp():
    c = Checkpointer()
    c.put("t1", "read_ledger", {"steps": 1, "findings": ["a"]})
    c.put("t1", "read_policy", {"steps": 2, "findings": ["a", "b"]})
    return c

check("a new thread has no checkpoint", lambda: Checkpointer().latest("nope") is None)
check("latest returns the most recent", lambda: _cp().latest("t1")["after"] == "read_policy")
check("history is ordered and complete", lambda: [c["seq"] for c in _cp().history("t1")] == [0, 1])
check("an earlier checkpoint is still reachable",
      lambda: _cp().at("t1", 0)["state"]["steps"] == 1)
check("checkpoints are snapshots, not references",
      lambda: (lambda c, s: (s["findings"].append("mutated"),
                             c.latest("t1")["state"]["findings"] == ["a", "b"])[1])(
              *(lambda: (lambda c: (c, c.threads["t1"][-1]["state"]))(_cp()))()) is not None)

## Section 2 &mdash; Resume after a crash

A graph that checkpoints can be killed and restarted. The test is that it continues rather than
repeating work already paid for.

In [ ]:
END = "__end__"

def run_graph(nodes, edges, conditions, entry, state, thread, cp,
              max_steps=8, stop_before=None, crash_after=None):
    """Run a graph, checkpointing after each node.

    stop_before  -- pause before this node and return, awaiting approval
    crash_after  -- simulate a pod death immediately after this node
    """
    resumed = cp.latest(thread)
    current = entry
    if resumed:
        state = resumed["state"]
        current = resumed["state"].get("__next__", entry)

    path = []
    while current != END:
        if state["steps"] >= max_steps:
            return merge(state, {"answer": "stopped: step budget"}), path, "budget"
        if stop_before and current == stop_before:
            cp.put(thread, "paused", {**state, "__next__": current})
            return state, path, "awaiting_approval"

        path.append(current)
        state = merge(state, nodes[current](state))
        nxt = conditions[current](state) if current in conditions else edges.get(current, END)
        cp.put(thread, current, {**state, "__next__": nxt})

        if crash_after and current == crash_after:
            return state, path, "crashed"
        current = nxt
    return state, path, "done"

def resume(nodes, edges, conditions, entry, thread, cp, **kw):
    """Continue a thread from its last checkpoint. Returns the same triple as run_graph."""
    last = cp.latest(thread)
    if last is None:
        raise ValueError("nothing to resume")
    return run_graph(nodes, edges, conditions, entry, BLANK, thread, cp, **kw)
                                     # TODO: which state should a resumed run start from?

In [ ]:
# --- Self-check: Section 2
def read_ledger(s):  return {"findings": [f"ledger: {lookup_payment(s['ref'])}"], "steps": s["steps"] + 1}
def read_policy(s):
    rec = LEDGER.get(s["ref"], {})
    return {"findings": [f"policy: {policy_for(rec.get('reason_code'))}"],
            "needs_human": rec.get("reason_code") in NEEDS_HUMAN, "steps": s["steps"] + 1}
def write_note(s):
    return {"answer": f"{s['ref']}: {'human decision required' if s['needs_human'] else 'operations may proceed'}",
            "steps": s["steps"] + 1}

REDUCERS = {"findings": lambda o, n: list(o or []) + list(n)}
def merge(state, update):
    out = dict(state)
    for k, v in update.items():
        out[k] = REDUCERS.get(k, lambda o, n: n)(state.get(k), v)
    return out

NODES = {"read_ledger": read_ledger, "read_policy": read_policy, "write_note": write_note}
EDGES = {"read_ledger": "read_policy", "read_policy": "write_note", "write_note": END}
FRESH = lambda ref="PMT-1005": {"ref": ref, "findings": [], "needs_human": False,
                                "steps": 0, "answer": None}

def _crash_then_resume():
    cp = Checkpointer()
    s1, p1, why1 = run_graph(NODES, EDGES, {}, "read_ledger", FRESH(), "t", cp,
                             crash_after="read_policy")
    s2, p2, why2 = resume(NODES, EDGES, {}, "read_ledger", "t", cp)
    return p1, why1, p2, why2, s2

check("the run crashes where we told it to",
      lambda: _crash_then_resume()[1] == "crashed")
check("it had completed two nodes before dying",
      lambda: _crash_then_resume()[0] == ["read_ledger", "read_policy"])
check("the resumed run finishes", lambda: _crash_then_resume()[3] == "done")
check("the resumed run does NOT repeat completed work",
      lambda: _crash_then_resume()[2] == ["write_note"],
      "start from the checkpointed state, not a fresh one")
check("findings from before the crash survived",
      lambda: len(_crash_then_resume()[4]["findings"]) == 2)
check("the final answer is correct after resuming",
      lambda: "human decision required" in _crash_then_resume()[4]["answer"])

## Section 3 &mdash; Pause for approval, and rewind

The same checkpoint mechanism, used two more ways. `write_note` is the irreversible step here, so
it is the one that waits for a human.

In [ ]:
def approve_and_continue(thread, cp):
    """A human approved the paused step. Continue from where it stopped."""
    return resume(NODES, EDGES, {}, "read_ledger", thread, cp)

def rewind(thread, cp, seq, changes: dict):
    """Rewind to checkpoint `seq`, apply `changes`, and re-run from there.

    Returns the new final state. The original history is left intact -- rewinding
    forks, it does not erase.
    """
    cp_at = cp.at(thread, seq)
    if cp_at is None:
        raise ValueError(f"no checkpoint {seq}")
    forked = Checkpointer()
    forked.threads[thread] = [c for c in cp.history(thread) if c["seq"] <= seq]
    state = {**cp_at["state"], **changes}
    forked.threads[thread][-1] = {**forked.threads[thread][-1], "state": state}
    return BLANK                     # TODO: re-run from the forked checkpointer and return
                                     # the final state only

In [ ]:
# --- Self-check: Section 3
def _paused():
    cp = Checkpointer()
    s, p, why = run_graph(NODES, EDGES, {}, "read_ledger", FRESH(), "t2", cp,
                          stop_before="write_note")
    return cp, s, p, why

check("the run pauses before the irreversible step",
      lambda: _paused()[3] == "awaiting_approval")
check("it paused with the reads already done", lambda: _paused()[2] == ["read_ledger", "read_policy"])
check("no answer was written while awaiting approval",
      lambda: _paused()[1]["answer"] is None,
      "the whole point of the gate is that the write has not happened yet")
check("approving continues to completion",
      lambda: approve_and_continue("t2", _paused()[0])[2] == "done")

def _rewound():
    cp = Checkpointer()
    run_graph(NODES, EDGES, {}, "read_ledger", FRESH(), "t3", cp)
    # an analyst disputes the human-decision flag and re-runs from after the policy read
    return cp, rewind("t3", cp, seq=1, changes={"needs_human": False})

check("rewinding with a changed field changes the outcome",
      lambda: "operations may proceed" in _rewound()[1]["answer"],
      "the original run said human decision required")
check("the original history is left intact",
      lambda: len(_rewound()[0].history("t3")) == 3,
      "rewinding forks; it must not erase what actually happened")

## Section 4 &mdash; The audit trail

The history is already the answer to *what did it know, and when?* Render it.

In [ ]:
def audit(thread, cp) -> str:
    """A human-readable trail: after each node, what was known and what came next."""
    rows = [f"{'seq':>4}  {'after':<14}{'steps':>6}{'findings':>10}  {'needs_human':<12}next"]
    rows.append("-" * 74)
    for c in cp.history(thread):
        s = c["state"]
        rows.append(f"{c['seq']:>4}  {c['after']:<14}{s.get('steps', 0):>6}"
                    f"{len(s.get('findings', [])):>10}  {str(s.get('needs_human')):<12}"
                    f"{s.get('__next__', '-')}")
    return "\n".join(rows)

try:
    _cpx = Checkpointer()
    run_graph(NODES, EDGES, {}, "read_ledger", FRESH(), "audit-demo", _cpx)
    print(audit("audit-demo", _cpx))
except NameError:
    print("(finish the sections above, then re-run this cell)")

In [ ]:
# --- Self-check: Section 4
def _audited():
    c = Checkpointer()
    run_graph(NODES, EDGES, {}, "read_ledger", FRESH(), "a1", c)
    return c

check("one checkpoint per node, plus a header and rule",
      lambda: len(audit("a1", _audited()).splitlines()) == 3 + 2)
check("the trail shows findings accumulating",
      lambda: "         1" in audit("a1", _audited()) and "         2" in audit("a1", _audited()))
check("the trail records when needs_human became true",
      lambda: audit("a1", _audited()).count("True") >= 2)
check("the trail is derived from recorded state, not regenerated",
      lambda: all("state" in c for c in _audited().history("a1")),
      "this is why it is stronger evidence than asking the model what it did")

## Run it for real

Have the model read your audit trail and answer the auditor's question. Note what it is doing:
reading a record, not recalling a run.

In [ ]:
if llm_ready():
    try:
        cp = Checkpointer()
        run_graph(NODES, EDGES, {}, "read_ledger", FRESH("PMT-1005"), "real", cp)
        trail = audit("real", cp)
        answer = ask(
            "You are answering an auditor. Using ONLY this execution trail, state what the system "
            "knew at the point it decided, and whether a human decision was required. If the trail "
            "does not support an answer, say so.\n\n" + trail
        )
        print(trail)
        print("\n--- the auditor's answer ---\n" + answer.strip()[:500])
    except NameError:
        print("(finish the sections above, then re-run this cell)")

### Read it

The model is summarising a **recorded** artefact. Ask it the same question with no trail and it
would produce something equally fluent and unfalsifiable &mdash; which is precisely the difference
Module 2 drew between reasoning text and evidence.

This is also the honest answer to &ldquo;can we explain what the agent did?&rdquo;. Not from the model.
From the checkpoints.

In [ ]:
score()

## Your turn

1. The checkpointer stores full state snapshots. On a long run that is a lot of duplication. Store
   diffs instead &mdash; then say what you lose when a diff chain has a gap.
2. State snapshots may contain the payment record. Which fields must never reach a checkpoint store
   under your own retention rules, and where would you enforce that &mdash; in the node, the reducer, or
   the checkpointer? Module 8 returns to this.